# setting up

In [1]:
import os
import numpy as np
import pandas as pd
from scipy.stats import zscore
from scipy import sparse
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_context('poster')

import scanpy as sc 

In [2]:
from SingleCellArchetype.main import SCA
from SingleCellArchetype.utils import plot_archetype

# Load data

In [3]:
%%time
f_anndata_in   = "../../data/v1_multiome/superdupermegaRNA_hasraw_multiome_L56IT.h5ad"
adata_all = sc.read(f_anndata_in)
adata_all

CPU times: user 572 ms, sys: 3.24 s, total: 3.81 s
Wall time: 18.3 s


AnnData object with n_obs × n_vars = 16948 × 16361
    obs: 'Age', 'Doublet', 'Doublet Score', 'n_counts', 'n_genes', 'percent_mito', 'sample', 'Type', 'Subclass', 'Class', 'Sample', 'total_counts', 'pct_counts_mt', 'n_genes_by_counts', 'total_counts_mt', 'Doublet?', 'Study', 'Type_leiden', 'time'
    var: 'feature_types'
    layers: 'norm'

In [4]:
# select samples
adata_all.obs['cond'] = adata_all.obs['sample'].apply(lambda x: x[:-1]) # .unique()

# remove mitocondria genes
adata_all = adata_all[:,~adata_all.var.index.str.contains(r'^mt-')]
adata_all = adata_all[:,~adata_all.var.index.str.contains(r'Xist')]

print(adata_all.obs['sample'].unique(), adata_all.obs['cond'].unique())

['P6c', 'P6a', 'P6b', 'P8c', 'P8b', ..., 'P17DRa', 'P21a', 'P21b', 'P21DRb', 'P21DRa']
Length: 25
Categories (25, object): ['P6a', 'P6b', 'P6c', 'P8a', ..., 'P21DRa', 'P21DRb', 'P21a', 'P21b'] ['P6' 'P8' 'P10' 'P12' 'P12DR' 'P14' 'P14DR' 'P17' 'P17DR' 'P21' 'P21DR']


# Prep data - select HVGs

In [5]:
conds = adata_all.obs['Age'].unique()
conds

['P6', 'P8', 'P10', 'P12', 'P12DR', ..., 'P14DR', 'P17', 'P17DR', 'P21', 'P21DR']
Length: 11
Categories (11, object): ['P6', 'P8', 'P10', 'P12', ..., 'P17', 'P17DR', 'P21', 'P21DR']

In [10]:
for age in conds:
    f_anndata_out  = f"../../data/v1_multiome/run_sca/multiome_l56it_{age}_hvg.h5ad"
    adata = adata_all[adata_all.obs['Age']==age].copy()


    # filter genes
    cond = np.ravel((adata.X>0).sum(axis=0)) > 10 # expressed in more than 10 cells
    adata = adata[:,cond].copy()
    genes = adata.var.index.values

    # counts
    x = adata.X
    cov = adata.obs['total_counts'].values

    # CP10k
    # xn = x/cov.reshape(x.shape[0], -1)*1e4
    xn = (sparse.diags(1/cov).dot(x))*1e4

    # log2(CP10k+1)
    # xln = xn.copy()
    # xln.data = np.log2(xln.data+1)

    log_xn = np.log2(1+np.array(xn.todense()))
    adata.layers[ 'lognorm'] = log_xn 
    adata.layers['zlognorm'] = zscore(log_xn, axis=0)

    # select HVGs with mean and var
    nbin = 20
    qth = 0.3

    # min
    gm = np.ravel(xn.mean(axis=0))

    # var
    tmp = xn.copy()
    tmp.data = np.power(tmp.data, 2)
    gv = np.ravel(tmp.mean(axis=0))-gm**2

    # cut 
    lbl = pd.qcut(gm, nbin, labels=np.arange(nbin))
    gres = pd.DataFrame()
    gres['name'] = genes
    gres['lbl'] = lbl
    gres['mean'] = gm
    gres['var'] = gv
    gres['ratio']= gv/gm

    # select
    gres_sel = gres.groupby('lbl')['ratio'].nlargest(int(qth*(len(gm)/nbin))) #.reset_index()
    gsel_idx = np.sort(gres_sel.index.get_level_values(1).values)
    assert np.all(gsel_idx != -1)

    adata_hvg = adata[:,gsel_idx]
    print(adata_hvg.shape)

    print(f_anndata_out)
    adata_hvg.write(f_anndata_out)

(2819, 4220)
../../data/v1_multiome/run_sca/multiome_l56it_P6_hvg.h5ad


/u/home/f/f7xiesnm/.conda/envs/napari/lib/python3.9/site-packages/anndata/_core/anndata.py:1230: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


(2371, 4400)
../../data/v1_multiome/run_sca/multiome_l56it_P8_hvg.h5ad


/u/home/f/f7xiesnm/.conda/envs/napari/lib/python3.9/site-packages/anndata/_core/anndata.py:1230: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


(1337, 4300)
../../data/v1_multiome/run_sca/multiome_l56it_P10_hvg.h5ad


...

/u/home/f/f7xiesnm/.conda/envs/napari/lib/python3.9/site-packages/anndata/_core/anndata.py:1230: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


(1013, 4200)
../../data/v1_multiome/run_sca/multiome_l56it_P21_hvg.h5ad


/u/home/f/f7xiesnm/.conda/envs/napari/lib/python3.9/site-packages/anndata/_core/anndata.py:1230: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c


(1916, 4200)
../../data/v1_multiome/run_sca/multiome_l56it_P21DR_hvg.h5ad


/u/home/f/f7xiesnm/.conda/envs/napari/lib/python3.9/site-packages/anndata/_core/anndata.py:1230: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  df[key] = c
